# Classification: CNN
- Used Pytorch pre trained ResNet-18
- Replace last layer for binary classification
- Used same '30kds' as YOLO, ViT (same data for training to later compare models)
- Resizes images to 224x224px as expected by the ResNet
- Runs on gpu/cpu

In [ ]:
# Run 'create_30k_ds.py' to create a balanced distribution (YOLO structure).
# This dataset has 30k real images and 30k fake images
# Split is 80% train, 10% test and 10% valid
# Structure:
'''
30kds/
│── train/
│   ├── Real/   (80% of real images)
│   ├── Fake/   (80% of selected fake images)
│
│── val/
│   ├── Real/   (10% of real images)
│   ├── Fake/   (10% of selected fake images)
│
│── test/
│   ├── Real/   (10% of real images)
│   ├── Fake/   (10% of selected fake images)
'''

In [1]:
import os
import torch
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision.datasets import ImageFolder
from tqdm import tqdm
import time
import random
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from PIL import Image
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

DATASET_PATH = "30kds"
TRAIN_PATH = os.path.join(DATASET_PATH, "train")
VAL_PATH = os.path.join(DATASET_PATH, "val")

# Define CSV log file
models_dir = "models"
os.makedirs(models_dir, exist_ok=True)  # Create directory if it doesn't exist
csv_filename = os.path.join(models_dir, "resnet_no_weights.csv")

# Initialize CSV log with error checking
try:
    with open(csv_filename, "w") as f:
        f.write("epoch,train_loss,val_loss,val_accuracy\n")
    print(f"Successfully created log file at {os.path.abspath(csv_filename)}")
except Exception as e:
    print(f"Error creating CSV file: {e}")
    # Try alternative location
    csv_filename = "resnet_no_weights.csv"  # Try in current directory instead
    with open(csv_filename, "w") as f:
        f.write("epoch,train_loss,val_loss,val_accuracy\n")

Using device: cuda
Successfully created log file at c:\Users\danie\Desktop\UNI-4ano\DL\models\resnet_no_weights.csv


In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet input size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = ImageFolder(root=TRAIN_PATH, transform=transform)
val_dataset = ImageFolder(root=VAL_PATH, transform=transform)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

In [3]:
# Define model custom ResNet-18 model
class CustomResNet(nn.Module):
    def __init__(self):
        super(CustomResNet, self).__init__()
        self.model = models.resnet18(weights=None)
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, 2)
    
    def forward(self, x):
        return self.model(x)

# Initialize model, loss, and optimizer
model = CustomResNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# Add this before training
print(f"Class to index mapping: {train_dataset.class_to_idx}")

epochs = 50
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]", leave=False)

    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        progress_bar.set_postfix(loss=f"{running_loss/len(train_loader):.4f}")

    train_loss = running_loss / len(train_loader)

    # Validation step
    model.eval()
    val_loss = 0.0
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = torch.softmax(model(images), dim=1)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    val_loss /= len(val_loader)
    val_acc = accuracy_score(y_true, y_pred)

    # Log results to CSV
    try:
        with open(csv_filename, "a") as f:
            log_line = f"{epoch+1},{train_loss:.5f},{val_loss:.5f},{val_acc:.5f}\n"
            f.write(log_line)
        # Verify file is being written correctly
        if epoch % 5 == 0:  # Check every 5 epochs
            with open(csv_filename, "r") as f:
                lines = f.readlines()
                print(f"CSV now has {len(lines)} lines")
    except Exception as e:
        print(f"Error writing to CSV at epoch {epoch+1}: {e}")

    print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # Adjust learning rate
    scheduler.step()

# Save model
torch.save(model.state_dict(), "models/resnet_no_weights.pth")
print("Training complete. Model saved.")

Class to index mapping: {'Fake': 0, 'Real': 1}


CSV now has 2 lines
Epoch [1/50] - Train Loss: 0.6943 | Val Loss: 0.6482 | Val Acc: 0.6871


Epoch [2/50] - Train Loss: 0.6759 | Val Loss: 0.5911 | Val Acc: 0.7548


Epoch [3/50] - Train Loss: 0.6669 | Val Loss: 0.6725 | Val Acc: 0.5788


Epoch [4/50] - Train Loss: 0.6601 | Val Loss: 0.5937 | Val Acc: 0.8323


...

CSV now has 47 lines
Epoch [46/50] - Train Loss: 0.0292 | Val Loss: 0.4486 | Val Acc: 0.8572


Epoch [47/50] - Train Loss: 0.0281 | Val Loss: 0.4522 | Val Acc: 0.8532


Epoch [48/50] - Train Loss: 0.0263 | Val Loss: 0.4643 | Val Acc: 0.8410


Epoch [49/50] - Train Loss: 0.0237 | Val Loss: 0.4619 | Val Acc: 0.8470


Epoch [50/50] - Train Loss: 0.0242 | Val Loss: 0.4511 | Val Acc: 0.8548
Training complete. Model saved.


In [ ]:
def classify_folder(model_path, folder_path, interval=2):
    """Classifies all images in a folder using a ResNet model and displays predictions."""
    
    class CustomResNet(nn.Module):
        def __init__(self):
            super(CustomResNet, self).__init__()
            self.model = models.resnet18(weights=None)
            num_ftrs = self.model.fc.in_features
            self.model.fc = nn.Linear(num_ftrs, 2)
        
        def forward(self, x):
            return self.model(x)

    # Use the same custom model architecture as during training
    model = CustomResNet()
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((224, 224)),  
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    class_labels = train_dataset.classes  # Make sure this matches your training classes order
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    random.shuffle(image_files)

    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)
        img = Image.open(image_path).convert("RGB")
        img_tensor = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            # Use softmax to get probabilities and ensure we're taking highest probability class
            output = model(img_tensor)
            probabilities = torch.softmax(output, dim=1)
            confidence, predicted_class = torch.max(probabilities, dim=1)
            predicted_class = predicted_class.item()
            confidence = confidence.item()

        predicted_label = class_labels[predicted_class]

        real_label = "Unknown"
        for class_name in class_labels:
            if class_name.lower() in image_file.lower():
                real_label = class_name
                break

        clear_output(wait=True)
        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.title(f"True: {real_label} | Pred: {predicted_label} (Conf: {confidence:.2f})", fontsize=14)
        plt.axis("off")
        display(plt.gcf())
        plt.close()

        time.sleep(interval)

# Run classification visualization
classify_folder("models/resnet_classifier.pth", "class_test")